# Triton Kernel for Fused Attention

In this tutorial, we build a **fused single-head attention** kernel using [Triton](https://triton-lang.org/), a Python-based language for writing highly efficient GPU kernels. We progress through three implementations:

1. **Naive PyTorch** — a straightforward, loop-based multi-head attention
2. **`torch.compile`** — leveraging PyTorch's graph compiler for automatic optimization
3. **Triton Fused Kernel** — a hand-written, tiled implementation using online softmax (FlashAttention-style)

Along the way, we explore the **online softmax** algorithm that makes it possible to fuse the entire attention computation into a single kernel pass over the K/V sequence, dramatically reducing memory traffic.

### Scope & Simplifications

The focus of this tutorial is **optimizing the attention computation for a single head**. In the MHA wrapper we still loop over heads in Python — we are *not* parallelizing across heads or batches on the GPU. In a production kernel you would add a batch and a head dimension to the launch grid so that every `(batch, head)` pair runs as its own program instance, eliminating the Python loop entirely.

We also assume that an entire block of queries and the full `head_dim` can fit in GPU SRAM (registers / shared memory) simultaneously. To make this feasible we keep `head_dim` small (here 32) by splitting `hidden_dim` across many heads. The tile size along the sequence axis (`BLOCK_SEQ`) is a tunable parameter — larger blocks amortize launch overhead but require more SRAM, so the right value depends on your GPU's resources.

### Setup & Configuration

We use a configuration typical of large language models: a sequence length of 2048, a hidden dimension of 4096, and 128 attention heads (giving a per-head dimension of 32).

In [76]:
import torch, math
import timeit

DEVICE = 'cuda:5'

SEQ_LEN = 2048
HIDDEN_DIM = 4096
NUM_HEADS = 128
HEAD_DIM = HIDDEN_DIM // NUM_HEADS

### Input & Weight Initialization

We create a random input tensor $x$ of shape `(seq_len, hidden_dim)` and four weight matrices ($W_Q, W_K, W_V, W_O$) of shape `(hidden_dim, hidden_dim)`. Everything is in `bfloat16` to match typical LLM training precision.

In [77]:
x = torch.randn(SEQ_LEN, HIDDEN_DIM, device=DEVICE, dtype=torch.bfloat16)
Wq = torch.randn(HIDDEN_DIM, HIDDEN_DIM, device=DEVICE, dtype=torch.bfloat16)
Wk = torch.randn(HIDDEN_DIM, HIDDEN_DIM, device=DEVICE, dtype=torch.bfloat16)
Wv = torch.randn(HIDDEN_DIM, HIDDEN_DIM, device=DEVICE, dtype=torch.bfloat16)
Wo = torch.randn(HIDDEN_DIM, HIDDEN_DIM, device=DEVICE, dtype=torch.bfloat16)

## Part 1: Multi-Head Attention in PyTorch

Multi-Head Attention (MHA) is the core building block of Transformer models. Given an input sequence $x$, we project it into queries ($Q$), keys ($K$), and values ($V$) using learned weight matrices, split them across multiple heads, compute scaled dot-product attention independently per head, and concatenate the results.

The standard scaled dot-product attention for a single head is:

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^T}{\sqrt{d_k}}\right) V$$

where $d_k$ is the dimension of each head.

### 1.1 Naive PyTorch Implementation

Our first implementation is deliberately simple: we compute attention head-by-head in a Python loop. This is easy to read but leaves significant performance on the table — the loop prevents parallelism across heads, and the intermediate attention matrix ($\text{seq\_len} \times \text{seq\_len}$) is fully materialized in GPU memory.

In [78]:
# x is input tensor of shape (batch_size, seq_len, hidden_dim)
def attention_head(q, k, v):
    seq_len = q.shape[-2]
    head_dim = q.shape[-1]
    attn_scores = q @ torch.transpose(k, -1, -2) / math.sqrt(head_dim) # (seq_len, seq_len)
    mask = torch.triu(torch.ones((seq_len, seq_len), device=q.device), diagonal=1).bool()
    attn_scores = torch.masked_fill(attn_scores, mask, float('-inf'))
    attn_output = torch.softmax(attn_scores, dim=-1) @ v # (seq_len, head_dim)
    return attn_output

def MHA_fwd(x, Wq, Wk, Wv, Wo, NUM_HEADS):
    # Wq, Wk, Wv are weight matrices of shape (hidden_dim, hidden_dim) 
    # q, k, v projections of x
    
    q = x @ Wq # (seq_len, hidden_dim)
    k = x @ Wk # (seq_len, hidden_dim)
    v = x @ Wv # (seq_len, hidden_dim)

    q = q.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)
    k = k.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)
    v = v.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)

    attn_output = torch.hstack([attention_head(q[i], k[i], v[i]) for i in range(NUM_HEADS)])

    return attn_output @ Wo # (seq_len, hidden_dim)


Let's benchmark the naive implementation over 100 iterations: Run it a couple of times to get a better estimate

In [79]:
timeit.timeit(lambda: MHA_fwd(x, Wq, Wk, Wv, Wo, NUM_HEADS), number=100)

2.083332146052271

### 1.2 Accelerating with `torch.compile`

PyTorch's `torch.compile` traces the computation graph and applies kernel fusion, operator rewriting, and other optimizations automatically. By compiling just the `attention_head` function, we can get a meaningful speedup with minimal code changes.

In [80]:
# x is input tensor of shape (batch_size, seq_len, hidden_dim)
compiled_attention_head = torch.compile(attention_head)

def compiled_MHA_fwd(x, Wq, Wk, Wv, Wo, NUM_HEADS):
    # Wq, Wk, Wv are weight matrices of shape (hidden_dim, hidden_dim) 
    # q, k, v projections of x
    
    q = x @ Wq # (seq_len, hidden_dim)
    k = x @ Wk # (seq_len, hidden_dim)
    v = x @ Wv # (seq_len, hidden_dim)

    q = q.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)
    k = k.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)
    v = v.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)

    attn_output = torch.hstack([compiled_attention_head(q[i], k[i], v[i]) for i in range(NUM_HEADS)])

    return attn_output @ Wo # (seq_len, hidden_dim)

In [81]:
timeit.timeit(lambda: compiled_MHA_fwd(x, Wq, Wk, Wv, Wo, NUM_HEADS), number=100)

1.5880393030820414

## Part 2: Understanding Softmax — From Safe to Online

Before writing the fused Triton kernel, we need to understand **online softmax**, the key algorithmic insight behind FlashAttention.

The standard softmax $\sigma(x)_i = \frac{e^{x_i}}{\sum_j e^{x_j}}$ is numerically unstable due to potential overflow in the exponentials. The **safe softmax** subtracts the maximum value first:

$$\sigma(x)_i = \frac{e^{x_i - \max(x)}}{\sum_j e^{x_j - \max(x)}}$$

Here's the safe softmax in PyTorch — we'll use this as our reference implementation:

In [82]:
def safe_softmax_torch(x):
    out = torch.exp(x - torch.max(x, dim=-1, keepdim=True).values)
    out = out / torch.sum(out, dim=-1, keepdim=True)
    return out

# safe_softmax_torch(x)

### 2.1 Scalar Safe Softmax vs. Online Softmax

Below we implement two scalar versions of softmax to illustrate the key algorithmic difference:

- **Safe softmax** requires **three passes** over the data: one to find the max, one to compute the sum of exponentials, and one to normalize.
- **Online softmax** requires only **two passes**: a single forward pass that maintains a running max and sum using a recurrence relation, followed by a normalization pass.

The online softmax recurrence updates the running sum when a new maximum is found:

$$\ell_{\text{new}} = \ell_{\text{old}} \cdot e^{(m_{\text{old}} - m_{\text{new}})} + e^{(x_i - m_{\text{new}})}$$

This is crucial for the fused attention kernel — we process K/V in tiles and need to update softmax statistics incrementally without ever materializing the full attention matrix.

In [83]:
def safe_softmax(x):
    x_max = float('-inf')
    x_sum = 0.0
    y = []
    for _x in x:
        x_max = max(_x, x_max)
    for _x in x:
        x_sum = x_sum + math.exp(_x - x_max)
    for i in range(len(x)):
        y.append(math.exp(x[i] - x_max) / x_sum)
    return y

# Recurrence relation for online softmax: sum = prev_sum * e^(old_max - max) + e^(x - max)
def online_softmax(x):
    _max = float('-inf')
    _sum = 0.0
    y = []
    for i in range(len(x)):
        _old_max = _max
        _max = max(_max, x[i])
        _sum = _sum * math.exp(_old_max - _max) + math.exp(x[i] - _max)
    for i in range(len(x)):
        y.append(math.exp(x[i] - _max) / _sum)
    return y

t = torch.randn(5).tolist()
safe_softmax(t), online_softmax(t), torch.softmax(torch.tensor(t), dim=-1)

([0.13527826550432429,
  0.11640806342863252,
  0.07591582961081986,
  0.5046486786441639,
  0.16774916281205948],
 [0.13527826550432426,
  0.11640806342863251,
  0.07591582961081986,
  0.5046486786441639,
  0.16774916281205945],
 tensor([0.1353, 0.1164, 0.0759, 0.5046, 0.1677]))

## Part 3: Fused Attention in Triton

Now we bring it all together. Our Triton kernel implements **FlashAttention-style fused attention**: instead of materializing the full $(\text{seq\_len} \times \text{seq\_len})$ attention matrix, we tile over the K/V sequence in blocks, computing partial attention scores and accumulating the output using online softmax.

**Why is this faster?** The naive approach writes the full attention matrix to HBM (GPU global memory) and reads it back — an $O(N^2)$ memory operation. The fused kernel keeps everything in SRAM (shared memory / registers), reducing memory traffic to $O(N)$.

### 3.1 The Fused Attention Forward Kernel

Triton programs operate on **blocks** of data rather than individual scalars (unlike CUDA, where each thread handles one element). Our kernel tiles the query rows and iterates over K/V tiles, performing the online softmax update at each step.

**Kernel structure:**
- Each program instance handles a block of `BLOCK_SEQ` query rows
- It loops over all K/V tiles of size `BLOCK_SEQ`
- At each tile: compute $Q \cdot K^T$, apply the online softmax update, accumulate $P \cdot V$
- After the loop: normalize by the final softmax denominator

In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def fused_attn_fwd_kernel(
    Q_ptr, K_ptr, V_ptr, O_ptr,
    stride_q_row, stride_qd,
    stride_k_row, stride_kd,
    stride_v_row, stride_vd,
    stride_o_row, stride_od,
    SEQ_LEN, HIDDEN_DIM,
    sm_scale,
    BLOCK_SEQ: tl.constexpr,
    BLOCK_HDIM: tl.constexpr,
):
    pid_m = tl.program_id(0)  # which block of query rows

    offs_q = pid_m * BLOCK_SEQ + tl.arange(0, BLOCK_SEQ) # query row indices
    offs_kv = tl.arange(0, BLOCK_SEQ) # key/value row indices
    offs_d = tl.arange(0, BLOCK_HDIM) # head dimension indices

    # Q block: [BLOCK_SEQ, BLOCK_HDIM]
    q_ptrs = Q_ptr + offs_q[:, None] * stride_q_row + offs_d[None, :] * stride_qd
    q_mask = (offs_q[:, None] < SEQ_LEN) & (offs_d[None, :] < HIDDEN_DIM)
    q = tl.load(q_ptrs, mask=q_mask, other=0.0)

    # running max and running denominator
    m_i = tl.full((BLOCK_SEQ,), float("-inf"), tl.float32)
    l_i = tl.zeros((BLOCK_SEQ,), dtype=tl.float32)

    # running numerator accumulator for output
    acc = tl.zeros((BLOCK_SEQ, BLOCK_HDIM), dtype=tl.float32)

    # loop over K/V tiles
    for start_m in range(0, SEQ_LEN, BLOCK_SEQ):
        k_idx = start_m + offs_kv

        # K block: [BLOCK_SEQ, BLOCK_HDIM]
        k_ptrs = K_ptr + k_idx[:, None] * stride_k_row + offs_d[None, :] * stride_kd
        k_mask = (k_idx[:, None] < SEQ_LEN) & (offs_d[None, :] < HIDDEN_DIM)
        k = tl.load(k_ptrs, mask=k_mask, other=0.0)

        # scores = Q @ K^T   => [BLOCK_M, BLOCK_N]
        scores = tl.dot(q, tl.trans(k)) * sm_scale

        # mask out out-of-bounds keys
        scores = tl.where(k_idx[None, :] < SEQ_LEN, scores, float("-inf"))

        # online softmax update
        m_ij = tl.max(scores, axis=1)                  # rowwise max of this tile
        m_new = tl.maximum(m_i, m_ij)

        correction_factor = tl.exp(m_i - m_new)       # rescale old stats
        p = tl.exp(scores - m_new[:, None])           # exp of shifted tile scores

        l_new = l_i * correction_factor + tl.sum(p, axis=1)

        # V block: [BLOCK_SEQ, BLOCK_HDIM]
        v_ptrs = V_ptr + k_idx[:, None] * stride_v_row + offs_d[None, :] * stride_vd
        v_mask = (k_idx[:, None] < SEQ_LEN) & (offs_d[None, :] < HIDDEN_DIM)
        v = tl.load(v_ptrs, mask=v_mask, other=0.0)

        # rescale old accumulator, then add current tile contribution
        acc = acc * correction_factor[:, None]
        acc = acc + tl.dot(p.to(v.dtype), v)

        m_i = m_new
        l_i = l_new

    # normalize at end
    out = acc / l_i[:, None]

    # store output
    o_ptrs = O_ptr + offs_q[:, None] * stride_o_row + offs_d[None, :] * stride_od
    o_mask = (offs_q[:, None] < SEQ_LEN) & (offs_d[None, :] < HIDDEN_DIM)
    tl.store(o_ptrs, out, mask=o_mask)

### 3.2 Python Wrapper

The wrapper function sets up the output tensor, computes the launch grid, and invokes the kernel. Note that `BLOCK_HDIM` is rounded up to the next power of 2 — a Triton requirement for efficient memory access patterns.

In [85]:
def fused_attention(q, k, v, sm_scale=None, BLOCK_SEQ=64):
    assert q.ndim == 2 and k.ndim == 2 and v.ndim == 2
    Mq, D = q.shape
    Mk, Dk = k.shape
    Mv, Dv = v.shape

    assert D == Dk == Dv
    assert Mq == Mk == Mv
    assert q.is_cuda and k.is_cuda and v.is_cuda

    if sm_scale is None:
        sm_scale = 1.0 / (D ** 0.5)

    o = torch.empty((Mq, D), device=q.device, dtype=torch.bfloat16)

    grid = (triton.cdiv(Mq, BLOCK_SEQ),)

    with torch.cuda.device(q.device):
        fused_attn_fwd_kernel[grid](
            q, k, v, o,
            q.stride(0), q.stride(1),
            k.stride(0), k.stride(1),
            v.stride(0), v.stride(1),
            o.stride(0), o.stride(1),
            Mq, D,
            sm_scale,
            BLOCK_SEQ=BLOCK_SEQ,
            BLOCK_HDIM=triton.next_power_of_2(D),
        )
    return o

### 3.3 Putting It Together: Triton-Powered MHA

Now we wire the fused attention kernel into our full MHA function, replacing the per-head PyTorch attention with our Triton kernel:

In [86]:
def mha_fwd_triton(x, Wq, Wk, Wv, Wo, NUM_HEADS):
    # Wq, Wk, Wv are weight matrices of shape (hidden_dim, hidden_dim) 
    # q, k, v projections of x
    
    q = x @ Wq # (seq_len, hidden_dim)
    k = x @ Wk # (seq_len, hidden_dim)
    v = x @ Wv # (seq_len, hidden_dim)

    q = q.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)
    k = k.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)
    v = v.view((SEQ_LEN, NUM_HEADS, HIDDEN_DIM // NUM_HEADS)).transpose(0, 1)

    attn_output = torch.hstack([
        fused_attention(
            q[i], k[i], v[i],
            sm_scale=None, BLOCK_SEQ=32,
        ) for i in range(NUM_HEADS)])

    return attn_output @ Wo # (seq_len, hidden_dim)

Let's benchmark the Triton fused attention and compare it to our earlier implementations:

In [87]:
timeit.timeit(lambda: mha_fwd_triton(x, Wq, Wk, Wv, Wo, NUM_HEADS), number=100)

0.7387639700900763

## Results Summary

| Implementation | Time (100 iters) | Relative Speedup |
|---|---|---|
| Naive PyTorch | ~2.08s | 1.0x |
| `torch.compile` | ~1.59s | ~1.3x |
| **Triton Fused Attention** | **~0.74s** | **~2.8x** |

The Triton fused attention kernel achieves a **~2.8x speedup** over naive PyTorch by eliminating the materialization of the full attention matrix and keeping intermediate results in fast on-chip memory. This is the same core idea behind [FlashAttention](https://arxiv.org/abs/2205.14135).

### Key Takeaways

- **Online softmax** enables single-pass attention computation without materializing $O(N^2)$ intermediate results
- **Triton's block-based programming model** maps naturally to tiled algorithms like FlashAttention
- Even without extensive tuning, a hand-written Triton kernel can significantly outperform both naive and `torch.compile`-optimized PyTorch
- For production use, consider `torch.nn.functional.scaled_dot_product_attention` which uses optimized backends (FlashAttention, Memory-Efficient Attention) under the hood